# Broodpile Pipeline

This notebook runs the broodpile workflow end to end using the existing repo scripts.

Steps:
1. Run YOLO segmentation and write raw `*.jsonl.gz` detections.
2. Detect colony masks and write `colonies.pkl`.
3. Filter raw YOLO JSONL detections into `*.filtered.jsonl.gz`.
4. Combine the filtered detections with colony masks and write per-colony CSVs.

The notebook uses strict inline constants for `width`, `height`, and `fps`, and processes only the `larvae` class downstream.

## Inputs
- `model_path`: YOLO segmentation weights.
- `runs`: four run definitions, each with a video input folder, prediction folder, filtered folder, and combine output folder.
- `image_path`: colony mask image used to extract colony polygons.
- `width`, `height`, `fps`: strict inline constants for this experiment.
- `colonies_pkl`: output file written by colony-mask processing.
- each run has its own `prediction_dir`, `filtered_dir`, and `output_dir`.
- each run also carries its own `video_dir` for preview mode.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

repo_dir = Path.cwd().resolve()
experiment_dir = Path("/media/AntGate/user/patrick/shared/broodpile_pipeline_output").expanduser().resolve()
input_dir = Path("/media/AntGate/user/patrick/shared/").expanduser().resolve()
python = sys.executable

# Required inputs.
model_path = experiment_dir / "best.pt"
image_path = experiment_dir / "mask.png"
colonies_pkl = experiment_dir / "colonies.pkl"

suffixes = ["patrick_oct17_alexa647_20241017_162401.40264048", "patrick_oct17_alexa647_20241017_162401.40264049", "patrick_oct17_alexa647_20241017_162401.40264052", "patrick_oct17_alexa647_20241017_162401.40264057"]
runs = [
    {
        "name": suffixes[0],
        "video_input_path": input_dir / suffixes[0],
        "prediction_dir": experiment_dir / suffixes[0] / "predictions",
        "filtered_dir": experiment_dir / suffixes[0] / "filter_predictions",
        "output_dir": experiment_dir / suffixes[0] / "combine_final",
        "video_dir": experiment_dir / suffixes[0],
    },
    {
        "name": suffixes[1],
        "video_input_path": input_dir / suffixes[1],
        "prediction_dir": experiment_dir / suffixes[1] / "predictions",
        "filtered_dir": experiment_dir / suffixes[1] / "filter_predictions",
        "output_dir": experiment_dir / suffixes[1] / "combine_final",
        "video_dir": experiment_dir / suffixes[1],
    },
    {
        "name": suffixes[2],
        "video_input_path": input_dir / suffixes[2],
        "prediction_dir": experiment_dir / suffixes[2] / "predictions",
        "filtered_dir": experiment_dir / suffixes[2] / "filter_predictions",
        "output_dir": experiment_dir / suffixes[2] / "combine_final",
        "video_dir": experiment_dir / suffixes[2],
    },
    {
        "name": suffixes[3],
        "video_input_path": input_dir / suffixes[3],
        "prediction_dir": experiment_dir / suffixes[3] / "predictions",
        "filtered_dir": experiment_dir / suffixes[3] / "filter_predictions",
        "output_dir": experiment_dir / suffixes[3] / "combine_final",
        "video_dir": experiment_dir / suffixes[3],
    },
]

orientation = "clockwise"
width = 4512
height = 4512
fps = 25.0
use_fixed_threshold = False
enable_preview = False
segmentation_jobs = 1
write_segmentation_overlay = False

# Filtering config passed directly to yolo_filtering.filter_one_file.
filter_cfg = {
    "paths": {
        "output_dir": "",
    },
    "iou_threshold": 0.3,
    "mask_buffer_px": 0.0,
    "class_thresholds": {
        0: 0.1,
    },
    "class_names": ["larvae"],
    "classes_to_process": [0],
    "containment_fraction": 0.5,
    "parallel": {
        "enabled": False,
        "workers": 1,
    },
    "overlay_colors": [
        [34, 139, 230],
    ],
    "test": {
        "enabled": enable_preview,
        "video_dir": "",
        "preview_fps": fps,
        "edge_thickness": 3,
    },
}

required_paths = [
    model_path,
    image_path,
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)
for run in runs:
    if not run["video_input_path"].exists():
        raise FileNotFoundError(run["video_input_path"])

for run in runs:
    run["prediction_dir"].mkdir(parents=True, exist_ok=True)
    run["filtered_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "repo_dir": str(repo_dir),
    "experiment_dir": str(experiment_dir),
    "model_path": str(model_path),
    "image_path": str(image_path),
    "colonies_pkl": str(colonies_pkl),
    "runs": [
        {
            "name": run["name"],
            "video_input_path": str(run["video_input_path"]),
            "prediction_dir": str(run["prediction_dir"]),
            "filtered_dir": str(run["filtered_dir"]),
            "output_dir": str(run["output_dir"]),
            "video_dir": str(run["video_dir"]),
        }
        for run in runs
    ],
    "width": width,
    "height": height,
    "fps": fps,
    "class_names": ["larvae"],
}, indent=2))


{
  "repo_dir": "/home/tracking/Dropbox @RU Dropbox/Juergen Wheeler/Jina_shared/scripts/broodpile",
  "experiment_dir": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output",
  "model_path": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output/best.pt",
  "image_path": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output/mask.png",
  "colonies_pkl": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output/colonies.pkl",
  "runs": [
    {
      "name": "patrick_oct17_alexa647_20241017_162401.40264048",
      "video_input_path": "/media/AntGate/user/patrick/shared/patrick_oct17_alexa647_20241017_162401.40264048",
      "prediction_dir": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/predictions",
      "filtered_dir": "/media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/filter_predictions",
      "output_dir": "/media/AntGate/user/patrick

## 1. Run segmentation

This calls `yolo_segment.sh` to generate raw `*.jsonl.gz` detections in `prediction_dir`.

In [15]:
for run in runs:
    cmd = [
        "bash",
        str(repo_dir / "yolo_segment.sh"),
        str(model_path),
        str(run["video_input_path"]),
        str(run["prediction_dir"]),
        str(segmentation_jobs),
    ]
    if write_segmentation_overlay:
        cmd.append(str(run["prediction_dir"] / "overlays"))

    subprocess.run(cmd, check=True, cwd=repo_dir)
    jsonl_files = sorted(run["prediction_dir"].glob("*.jsonl.gz"))
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl.gz files were generated in {run['prediction_dir']}")
    print(f"[{run['name']}] Wrote {len(jsonl_files)} raw prediction files to {run['prediction_dir']}")


Traceback (most recent call last):
  File "/home/tracking/Dropbox @RU Dropbox/Juergen Wheeler/Jina_shared/scripts/broodpile/yolo_segment.py", line 10, in <module>
    from ultralytics import YOLO
  File "<frozen importlib._bootstrap>", line 1075, in _handle_fromlist
  File "/home/tracking/.local/lib/python3.10/site-packages/ultralytics/__init__.py", line 38, in __getattr__
    return getattr(importlib.import_module("ultralytics.models"), name)
  File "/usr/lib/python3.10/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
  File "/home/tracking/.local/lib/python3.10/site-packages/ultralytics/models/__init__.py", line 6, in <module>
    from .sam import SAM
  File "/home/tracking/.local/lib/python3.10/site-packages/ultralytics/models/sam/__init__.py", line 3, in <module>
    from .model import SAM
  File "/home/tracking/.local/lib/python3.10/site-packages/ultralytics/models/sam/model.py", line 24, in <module>
    from .predi

KeyboardInterrupt: 

## 2. Build colony masks

This calls `process_colony_masks.py` directly. It writes `colonies.pkl` and the labeled mask overlay next to the source mask image.

In [ ]:
cmd = [
    python,
    str(repo_dir / "process_colony_masks.py"),
    "--image-path", str(image_path),
    "--colonies-file", str(colonies_pkl),
    "--orientation", orientation,
    "--width", str(width),
    "--height", str(height),
]
if use_fixed_threshold:
    cmd.append("--fixed-threshold")

subprocess.run(cmd, check=True, cwd=repo_dir)
if not colonies_pkl.exists():
    raise FileNotFoundError(colonies_pkl)
print(f"Wrote {colonies_pkl}")


## 3. Filter detections

This imports `yolo_filtering.py` and runs `filter_one_file` directly so the notebook can pass a strict config dictionary without relying on a hard-coded config file path.

In [ ]:
from yolo_filtering import filter_one_file, filtered_out_name

for run in runs:
    filter_cfg["paths"]["output_dir"] = str(run["filtered_dir"])
    filter_cfg["test"]["video_dir"] = str(run["video_dir"])
    jsonl_files = sorted([p for p in run["prediction_dir"].iterdir() if p.name.endswith(".jsonl.gz")])
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl.gz files found in {run['prediction_dir']}")

    for jsonl_path in jsonl_files:
        out_path = run["filtered_dir"] / filtered_out_name(jsonl_path)
        filter_one_file(jsonl_path, out_path, filter_cfg)
        if not out_path.exists():
            raise FileNotFoundError(out_path)

    print(f"[{run['name']}] Filtered {len(jsonl_files)} files into {run['filtered_dir']}")


## 4. Combine colony metrics

This calls `combine_segmentation_colony.py` for each run and writes one CSV set per output directory.

In [ ]:
for run in runs:
    cmd = [
        python,
        str(repo_dir / "combine_segmentation_colony.py"),
        str(run["filtered_dir"]),
        str(colonies_pkl),
        "-o", str(run["output_dir"]),
    ]
    if enable_preview:
        cmd.extend(["--preview", "--video_dir", str(run["video_dir"]), "--preview_fps", str(fps)])

    subprocess.run(cmd, check=True, cwd=repo_dir)
    print(f"[{run['name']}] Wrote outputs to {run['output_dir']}")


## Outputs

The pipeline should now have:
- `colonies.pkl`
- raw `*.jsonl.gz` files in each run prediction directory
- `*.filtered.jsonl.gz` files in each run filtered directory
- `colony_XX_detections.csv` files in each run combine output directory

In [ ]:
print("colonies.pkl:", colonies_pkl.exists())
for run in runs:
    print(f"[{run['name']}] raw jsonl files:", len(list(run["prediction_dir"].glob("*.jsonl.gz"))))
    print(f"[{run['name']}] filtered files:", len(list(run["filtered_dir"].glob("*.filtered.jsonl.gz"))))
    print(f"[{run['name']}] csv files:", len(list(run["output_dir"].glob("colony_*_detections.csv"))))
